# Part 3: Advanced Modeling - Ensembles, Tuning, and Full ML Pipeline

## Objective

Now that I have a working regression and classification model from Part 2,
this part is about checking if fancier models (decision trees, random forest,
gradient boosting) actually do better, tuning them properly instead of just
guessing hyperparameters, and packaging the best one into a pipeline I can
save and reload later.

This covers:
- Decision tree baseline (unconstrained vs controlled)
- Gini vs entropy comparison
- Random Forest + feature importance
- Gradient Boosting + a feature ablation check
- Cross validated comparison of everything
- GridSearchCV tuning
- Manual learning curve
- Saving the final model with joblib


## Step 1: Rebuild the preprocessed data + Decision Tree baseline

- New notebook, so I need to redo the same encoding from Part 2 first to get
X_train_scaled, X_test_scaled, y_clf_train, y_clf_test back before I can use
them here - a new Colab session doesn't remember variables from the old
notebook. Copy-pasting the exact same preprocessing block from Part 2 so the
numbers stay consistent.

- Then training a plain DecisionTreeClassifier with no restrictions at all
(max_depth=None means it keeps splitting until every leaf is pure). Checking
train accuracy vs test accuracy right after to see how badly it overfits.

In [26]:
# Importing pandas
import pandas as pd

# Uploading cleaned_data.csv from Part-1
from google.colab import files
files.upload()

# Printing a simple confirmation message
print("\nDataset loaded successfully!")

# Printing dataset shape
print("\nDataset Shape:", df.shape)

# Printing only the first 5 rows
print("\nFirst 5 rows of the dataset:")
print(df.head())

Saving cleaned_data.csv to cleaned_data (4).csv

Dataset loaded successfully!

Dataset Shape: (1460, 81)

First 5 rows of the dataset:
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl  

In [27]:
# Importing all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib

# Loading the cleaned dataset
# ===============================

df = pd.read_csv("cleaned_data.csv")

# Regression target
y_reg = df["SalePrice"]

# Classification target
# Houses above the median price become class 1, others become class 0
y_clf = (y_reg > y_reg.median()).astype(int)

# Creating feature matrix
# ===============================

# Removing Id and SalePrice because they are not input features
X = df.drop(columns=["SalePrice", "Id"])

# MSSubClass is actually a category, not a continuous number
X["MSSubClass"] = X["MSSubClass"].astype(str)

# Handling categorical missing values
# ===============================

cat_cols = X.select_dtypes(include="object").columns.tolist()
for col in cat_cols:

    # Electrical has only a normal missing value, so I filled it using the most common value
    if col == "Electrical":
        X[col] = X[col].fillna(X[col].mode()[0])

    # Most other missing values mean the feature does not exist, so I used "None" as a category
    else:
        X[col] = X[col].fillna("None")

# Encoding ordinal columns
# ===============================

quality_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

ordinal_maps = {

    "ExterQual": quality_map,
    "ExterCond": quality_map,
    "BsmtQual": quality_map,
    "BsmtCond": quality_map,
    "HeatingQC": quality_map,
    "KitchenQual": quality_map,
    "FireplaceQu": quality_map,
    "GarageQual": quality_map,
    "GarageCond": quality_map,
    "PoolQC": quality_map,

    "BsmtExposure": {
        "None": 0,
        "No": 1,
        "Mn": 2,
        "Av": 3,
        "Gd": 4
    },

    "BsmtFinType1": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },

    "BsmtFinType2": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },

    "GarageFinish": {
        "None": 0,
        "Unf": 1,
        "RFn": 2,
        "Fin": 3
    },

    "Functional": {
        "Sal": 0,
        "Sev": 1,
        "Maj2": 2,
        "Maj1": 3,
        "Mod": 4,
        "Min2": 5,
        "Min1": 6,
        "Typ": 7
    },

    "LandSlope": {
        "Sev": 0,
        "Mod": 1,
        "Gtl": 2
    },

    "LotShape": {
        "IR3": 0,
        "IR2": 1,
        "IR1": 2,
        "Reg": 3
    },

    "Utilities": {
        "ELO": 0,
        "NoSeWa": 1,
        "NoSewr": 2,
        "AllPub": 3
    },

    "PavedDrive": {
        "N": 0,
        "P": 1,
        "Y": 2
    },

    "Fence": {
        "None": 0,
        "MnWw": 1,
        "GdWo": 2,
        "MnPrv": 3,
        "GdPrv": 4
    }
}


# Applying ordinal encoding

for col, mapping in ordinal_maps.items():
    X[col] = X[col].map(mapping)

# One-hot encoding nominal columns
# ===============================

nominal_cols = [
    col for col in cat_cols
    if col not in ordinal_maps
]

X = pd.get_dummies(
    X,
    columns=nominal_cols,
    drop_first=True
)

# Splitting the data
# ===============================

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X,
    y_reg,
    y_clf,
    test_size=0.2,
    random_state=42
)

# Scaling the features
# ===============================

scaler = StandardScaler()

# Fitting only on training data to avoid data leakage
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Final feature shape:", X.shape)

# Decision Tree (Default)
# ===============================

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train_scaled, y_clf_train)

# Predicting on both training and testing data
train_pred = tree_full.predict(X_train_scaled)
test_pred = tree_full.predict(X_test_scaled)

# Calculating accuracy
train_acc = accuracy_score(y_clf_train, train_pred)
test_acc = accuracy_score(y_clf_test, test_pred)


print("\nDecision Tree (Default)")
print("-" * 35)
print("Training Accuracy :", train_acc)
print("Testing Accuracy  :", test_acc)

Final feature shape: (1460, 215)

Decision Tree (Default)
-----------------------------------
Training Accuracy : 1.0
Testing Accuracy  : 0.8904109589041096


# Step 2: Training a controlled Decision Tree

- The first Decision Tree gave me 100% training accuracy, which usually means it learned the training data too well. This is called overfitting because the model performs very well on training data but not as well on new data.

- Now I am training another Decision Tree with some limits. I am using `max_depth=5` so the tree cannot grow too deep, and `min_samples_split=20` so it only creates a new split when there are enough samples.

- After that, I am also comparing Gini and Entropy to see if changing the splitting method makes any difference in the model's performance.

In [28]:
# Controlled Decision Tree
# ===============================

# Limiting the tree so it does not become too complex
tree_ctrl = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

# Training the model
tree_ctrl.fit(X_train_scaled, y_clf_train)

# Predicting on training and testing data
train_pred_ctrl = tree_ctrl.predict(X_train_scaled)
test_pred_ctrl = tree_ctrl.predict(X_test_scaled)

# Calculating accuracy
train_acc_ctrl = accuracy_score(y_clf_train, train_pred_ctrl)
test_acc_ctrl = accuracy_score(y_clf_test, test_pred_ctrl)

# Difference between training and testing accuracy
gap = train_acc_ctrl - test_acc_ctrl

print("Controlled Decision Tree")
print("-" * 35)
print(f"Training Accuracy : {train_acc_ctrl:.4f}")
print(f"Testing Accuracy  : {test_acc_ctrl:.4f}")
print(f"Accuracy Gap      : {gap:.4f}")

# Gini vs Entropy Comparison
# ===============================

# Decision Tree using Gini
tree_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

tree_gini.fit(X_train_scaled, y_clf_train)

gini_acc = accuracy_score(
    y_clf_test,
    tree_gini.predict(X_test_scaled)
)

# Decision Tree using Entropy
tree_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

tree_entropy.fit(X_train_scaled, y_clf_train)

entropy_acc = accuracy_score(
    y_clf_test,
    tree_entropy.predict(X_test_scaled)
)

print("\nGini vs Entropy")
print("-" * 35)
print(f"Gini Test Accuracy    : {gini_acc:.4f}")
print(f"Entropy Test Accuracy : {entropy_acc:.4f}")

Controlled Decision Tree
-----------------------------------
Training Accuracy : 0.9307
Testing Accuracy  : 0.8801
Accuracy Gap      : 0.0505

Gini vs Entropy
-----------------------------------
Gini Test Accuracy    : 0.8938
Entropy Test Accuracy : 0.8836


## Step 3: Random Forest

In the last step I used only one Decision Tree. It worked well, but one tree can
still overfit because it learns from only one version of the training data.
So now I am using a Random Forest.

- Instead of training one tree, Random Forest trains many trees and combines all
their predictions. Every tree is trained on a different random sample of the
training data, and every split checks only a random group of features. Because
of this, all the trees are a little different and they do not make exactly the
same mistakes.

- After training the model, I checked the training accuracy, testing accuracy and ROC-AUC score.

I also looked at the feature importance values to see which features helped the
model the most while making predictions.

In [29]:
# Random Forest
# ===============================

# Creating the Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Training the model
rf.fit(X_train_scaled, y_clf_train)

# Making predictions
train_pred_rf = rf.predict(X_train_scaled)
test_pred_rf = rf.predict(X_test_scaled)

# Calculating accuracy
rf_train_acc = accuracy_score(y_clf_train, train_pred_rf)
rf_test_acc = accuracy_score(y_clf_test, test_pred_rf)

# Getting prediction probabilities for ROC-AUC
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]
rf_auc = roc_auc_score(y_clf_test, rf_proba)

print("Random Forest")
print("-" * 35)
print(f"Training Accuracy : {rf_train_acc:.4f}")
print(f"Testing Accuracy  : {rf_test_acc:.4f}")
print(f"ROC-AUC Score     : {rf_auc:.4f}")

# Top 5 Important Features
# ===============================

# Creating a dataframe of feature importance values
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

# Sorting from highest importance to lowest
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 5 Important Features")
print("-" * 35)
print(feature_importance.head(5))

Random Forest
-----------------------------------
Training Accuracy : 0.9949
Testing Accuracy  : 0.9418
ROC-AUC Score     : 0.9834

Top 5 Important Features
-----------------------------------
        Feature  Importance
25    GrLivArea    0.083388
5   OverallQual    0.056604
28     FullBath    0.053926
7     YearBuilt    0.049113
12     BsmtQual    0.049090


## Step 4: Gradient Boosting and Feature Ablation

In this step, I trained a Gradient Boosting model. Unlike Random Forest, Gradient Boosting builds one tree at a time, where each new tree tries to correct the mistakes made by the previous trees.

- I used 100 trees, a learning rate of 0.1, and a maximum depth of 3.

- After training the model, I performed a feature ablation study to check whether the least important features were actually useful.

- I identified the 5 features with the lowest importance from the Random Forest model, removed them, trained another Random Forest with the same settings, and compared the ROC-AUC scores of both models.

In [30]:
# Gradient Boosting Classifier
# ===============================

# Creating the Gradient Boosting model
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Training the model
gb.fit(X_train_scaled, y_clf_train)

# Making predictions
gb_train_pred = gb.predict(X_train_scaled)
gb_test_pred = gb.predict(X_test_scaled)

# Predicting probabilities for ROC-AUC
gb_test_proba = gb.predict_proba(X_test_scaled)[:, 1]

# Calculating performance
gb_train_acc = accuracy_score(y_clf_train, gb_train_pred)
gb_test_acc = accuracy_score(y_clf_test, gb_test_pred)
gb_auc = roc_auc_score(y_clf_test, gb_test_proba)

print("Gradient Boosting")
print("-" * 35)
print(f"Training Accuracy : {gb_train_acc:.4f}")
print(f"Testing Accuracy  : {gb_test_acc:.4f}")
print(f"ROC-AUC Score     : {gb_auc:.4f}")


# Feature Ablation Study
# ===============================

# Getting the 5 least important features from Random Forest
lowest5 = feature_importance.tail(5)["Feature"].tolist()

print("\nLowest 5 Important Features")
print("-" * 35)

for feature in lowest5:
    print(feature)

# Converting scaled arrays back to DataFrames
# so I can remove columns by their names
X_train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_df = pd.DataFrame(X_test_scaled, columns=X.columns)

# Removing the lowest importance features
X_train_reduced = X_train_df.drop(columns=lowest5)
X_test_reduced = X_test_df.drop(columns=lowest5)

# Training another Random Forest with the reduced dataset
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_reduced, y_clf_train)

# Calculating ROC-AUC
rf_reduced_proba = rf_reduced.predict_proba(X_test_reduced)[:, 1]
rf_reduced_auc = roc_auc_score(y_clf_test, rf_reduced_proba)

print("\nFeature Ablation Results")
print("-" * 35)
print(f"Full Model AUC    : {rf_auc:.4f}")
print(f"Reduced Model AUC : {rf_reduced_auc:.4f}")

# Small conclusion based on the results
print("\nConclusion")
print("-" * 35)

if rf_reduced_auc >= rf_auc:
    print("The reduced model performed almost the same (or slightly better).")
    print("This suggests the removed features were contributing very little.")
    print("Removing them can make the model a little simpler without hurting performance.")
else:
    print("The reduced model performed worse after removing the features.")
    print("This means those features were still helping the model.")

Gradient Boosting
-----------------------------------
Training Accuracy : 0.9880
Testing Accuracy  : 0.9521
ROC-AUC Score     : 0.9863

Lowest 5 Important Features
-----------------------------------
Electrical_Mix
SaleType_ConLI
MiscFeature_TenC
SaleCondition_AdjLand
SaleType_Oth

Feature Ablation Results
-----------------------------------
Full Model AUC    : 0.9834
Reduced Model AUC : 0.9861

Conclusion
-----------------------------------
The reduced model performed almost the same (or slightly better).
This suggests the removed features were contributing very little.
Removing them can make the model a little simpler without hurting performance.


Conclusion...
-----------------------------------
- The reduced model performed almost the same (or slightly better).
- This suggests the removed features were contributing very little.
- Removing them can make the model a little simpler without hurting performance.

## Step 5: Comparing models using 5-Fold Cross Validation

Until now, I checked every model using only one train-test split.
The problem with a single split is that the results can change depending on which rows go into the training set and which go into the testing set.

- **To get a more reliable estimate, I used 5-Fold Stratified Cross Validation.**

- The data is divided into 5 parts. The model trains on 4 parts and tests on the remaining part. This process repeats 5 times, and then the average ROC-AUC score is calculated.

Finally, I compared four models:

- Logistic Regression
- Controlled Decision Tree
- Random Forest
- Gradient Boosting

I reported both the mean ROC-AUC and the standard deviation for each model.

In [31]:
# 5-Fold Cross Validation
# ===============================

# Creating Stratified K-Fold object
# This keeps the class balance almost the same in every fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Models to compare
models_cv = {
    "Logistic Regression": LogisticRegression(max_iter=2000),

    "Decision Tree (Depth = 5)": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

print("5-Fold Cross Validation Results")
print("-" * 60)

cv_results = {}

# Running cross validation for every model
for name, model in models_cv.items():

    scores = cross_val_score(
        model,
        X_train_scaled,
        y_clf_train,
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results[name] = (scores.mean(), scores.std())

    print(
        f"{name:<28}"
        f"Mean AUC : {scores.mean():.4f}   "
        f"Std : {scores.std():.4f}"
    )

5-Fold Cross Validation Results
------------------------------------------------------------
Logistic Regression         Mean AUC : 0.9525   Std : 0.0113
Decision Tree (Depth = 5)   Mean AUC : 0.9233   Std : 0.0148
Random Forest               Mean AUC : 0.9784   Std : 0.0051
Gradient Boosting           Mean AUC : 0.9749   Std : 0.0078


## Step 6: Hyperparameter Tuning and Manual Learning Curve

In this step, I used GridSearchCV to find the best Random Forest settings instead of choosing them myself.

I created a pipeline with:

- SimpleImputer
- StandardScaler
- Random Forest Classifier

The pipeline handles all preprocessing automatically, so I used the original training data without scaling it first.

I tested different values for:

- Number of trees
- Maximum tree depth
- Minimum samples in each leaf

This gave **18 different parameter combinations**.

- Since I used **5-fold cross validation**, the total number of model fits was:  **18 × 5 = 90**

- After finding the best model, I trained it using different amounts of the training data (20%, 40%, 60%, 80% and 100%).

For each training size, I calculated both the training ROC-AUC and the testing ROC-AUC to see whether the model improves when more data is available.

In [32]:
# Hyperparameter Tuning using GridSearchCV
# =========================================

# Creating a pipeline
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# Different parameter values to test
param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

# Creating 5-Fold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Running Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

# Training Grid Search
grid_search.fit(X_train, y_clf_train)

print("Best Parameters")
print("-" * 35)
print(grid_search.best_params_)

print("\nBest Cross Validation AUC")
print("-" * 35)
print(f"{grid_search.best_score_:.4f}")

# Saving the best pipeline
best_pipeline = grid_search.best_estimator_


# Manual Learning Curve
# =========================================

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for fraction in fractions:

    # Taking only part of the training data
    n = int(fraction * len(X_train))

    X_subset = X_train.iloc[:n]
    y_subset = y_clf_train.iloc[:n]

    # Training the best pipeline
    best_pipeline.fit(X_subset, y_subset)

    # Training ROC-AUC
    train_auc = roc_auc_score(
        y_subset,
        best_pipeline.predict_proba(X_subset)[:, 1]
    )

    # Testing ROC-AUC
    test_auc = roc_auc_score(
        y_clf_test,
        best_pipeline.predict_proba(X_test)[:, 1]
    )

    results.append([fraction, train_auc, test_auc])

# Showing the learning curve table
learning_curve = pd.DataFrame(
    results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Testing AUC"
    ]
)

print("\nManual Learning Curve")
print("-" * 35)
print(learning_curve)

# Training the best model again using the full training data
best_pipeline.fit(X_train, y_clf_train)
print("\nBest pipeline trained successfully.")

Best Parameters
-----------------------------------
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best Cross Validation AUC
-----------------------------------
0.9796

Manual Learning Curve
-----------------------------------
   Training Fraction  Training AUC  Testing AUC
0                0.2           1.0     0.980394
1                0.4           1.0     0.982267
2                0.6           1.0     0.984069
3                0.8           1.0     0.982718
4                1.0           1.0     0.985136

Best pipeline trained successfully.


## Step 7: Saving and Reloading the Best Model

In this final step, I saved the best pipeline using **Joblib**.

- Saving the model means I do not need to train it again every time I want to use it.

- After saving the model, I loaded it back into memory and tested it on two hand-crafted examples to make sure the saved file works correctly.

Finally, I will compare all the models from Part 2 and Part 3 in the README and recommend the best model based on the overall performance.

In [34]:
# Saving and Reloading the Best Model
# ===============================

import joblib

# Saving the best trained pipeline
joblib.dump(best_pipeline, "best_model.pkl")

print("best_model.pkl saved successfully!!")


# Loading the saved model
loaded_model = joblib.load("best_model.pkl")

print("Model loaded successfully!!")


# Creating two hand-crafted test rows
sample_rows = pd.DataFrame(
    [
        X_test.iloc[0].to_dict(),
        X_test.iloc[1].to_dict()
    ]
)

# Making predictions
predictions = loaded_model.predict(sample_rows)
probabilities = loaded_model.predict_proba(sample_rows)[:, 1]

print("\nPredictions")
print(predictions)

print("\nPrediction Probabilities")
print(probabilities)


# Downloading the saved model
from google.colab import files

files.download("best_model.pkl")

best_model.pkl saved successfully!!
Model loaded successfully!!

Predictions
[0 1]

Prediction Probabilities
[0.03 1.  ]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>